# News vector RAG — interactive tour

The deep **`AgensgraphVector`** showcase: real news (CC-News) in a pgvector HNSW
store with a keyword (`tsvector`) index, queried five ways — ending in a LangChain
**LCEL RAG chain**.

**Prerequisite — ingest first** (or with `DEMOS_FAKE=1`, the 50-article fixture):

```bash
cd langchain
NEWS_LIMIT=10000 .venv/bin/python examples/demos/03_news_vector_rag/ingest.py
```

In [ ]:
import pathlib
import sys

HERE = pathlib.Path.cwd()                  # .../03_news_vector_rag
p = HERE
while p != p.parent and not (p / "_common").is_dir():
    p = p.parent
sys.path.insert(0, str(p))                 # demos root, for _common
sys.path.insert(0, str(HERE))              # this dir, for `rag`

import pandas as pd
import rag                                  # store construction + helpers live here
from _common import agens, config
from _common.models import get_llm
from langchain_agensgraph.vectorstores import SearchType

vector = rag.open_store(SearchType.VECTOR)  # vector + filtered search
# vector + keyword, fused by reciprocal rank, the keyword ranking weighted twice
hybrid = rag.open_store(SearchType.HYBRID, fusion_weights=(1.0, 2.0))
graph = agens.make_graph("news")


def hits_df(hits, score="relevance"):
    return pd.DataFrame([
        {score: round(s, 3), "title": d.metadata.get("title", "")[:55],
         "domain": d.metadata.get("domain", ""), "date": d.metadata.get("date", "")}
        for d, s in hits
    ])


print("connected to", config.url().split("@")[-1])

## The corpus

In [ ]:
n = graph.query('MATCH (n:"Article") RETURN count(*) AS c')[0]["c"]
rng = graph.query(
    'MATCH (n:"Article") WHERE n.date IS NOT NULL '
    "RETURN min(n.date::text) AS lo, max(n.date::text) AS hi"
)[0]
print(f"{n:,} chunks · dates {rng['lo']} .. {rng['hi']}")
pd.DataFrame(graph.query(
    'MATCH (n:"Article") RETURN n.domain AS domain, count(*) AS chunks '
    "ORDER BY chunks DESC LIMIT 8"
))

## (a) Vector semantic search (HNSW)

In [ ]:
# Relevance between 0 and 1; similarity_search_with_score gives the raw cosine distance.
hits_df(vector.similarity_search_with_relevance_scores(
    "artificial intelligence and machine learning", k=5))

## (b) Metadata-filtered search

MongoDB-style filter operators on the chunk metadata.

In [ ]:
domains = [r["domain"] for r in graph.query(
    'MATCH (n:"Article") RETURN n.domain AS domain, count(*) AS c ORDER BY c DESC LIMIT 3')]
# A date the record did not carry is absent rather than "", so IS NOT NULL is the whole test.
lo = graph.query(
    'MATCH (n:"Article") WHERE n.date IS NOT NULL RETURN min(n.date::text) AS lo')[0]["lo"]
flt = {"$and": [{"domain": {"$in": domains}}, {"date": {"$gte": lo}}]}
print("filter =", flt)
hits_df(vector.similarity_search_with_relevance_scores("business and technology", k=5,
                                                       filter=flt))

## (c) Hybrid search (vector + keyword, RRF fusion)

`fusion_weights=(vector, keyword)` tunes the reciprocal-rank fusion; the score is the fused
one, larger being better. `hybrid_search_with_details` also shows each half's rank, and a
filter narrows both halves.

In [ ]:
display(hits_df(hybrid.similarity_search_with_score("stock market", k=5), score="fused"))
pd.DataFrame([
    {"fused": round(h.fused_score, 4), "vector rank": h.vector_rank,
     "keyword rank": h.keyword_rank, "title": h.document.metadata.get("title", "")[:55]}
    for h in hybrid.hybrid_search_with_details(
        "stock market", k=5, filter={"date": {"$gte": lo}})
])

## (d) `ef_search` — pgvector's recall control

`hnsw.ef_search` is how many candidates the index considers before ranking them, so
raising it trades latency for recall. Over-fetching `k` does not: a larger `LIMIT`
returns the same rows, because the index has already stopped looking. It is a setting of
the store, so each value below is a store of its own over the same chunks.

In [ ]:
for ef in (20, 200):
    print(f"hnsw.ef_search = {ef}")
    tuned = rag.open_store(SearchType.VECTOR, ef_search=ef)
    display(hits_df(tuned.similarity_search_with_relevance_scores(
        "sports", k=5, filter={"date": {"$gte": lo}})))

## (e) RAG — `as_retriever()` in an LCEL chain

The vector store becomes a LangChain retriever, composed with a prompt + LLM
into a cited, grounded answer.

In [7]:
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

retriever = vector.as_retriever(search_kwargs={"k": 5})

def format_docs(docs):
    return "\n\n".join(
        f"[{d.metadata.get('domain','?')} {d.metadata.get('date','')}] "
        f"{d.metadata.get('title','')}\n{d.page_content}" for d in docs)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY the news snippets below. Cite the source "
               "domains you rely on. If they don't cover it, say so."),
    ("human", "Question: {question}\n\nNews snippets:\n{context}"),
])
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | get_llm() | StrOutputParser()
)

print(chain.invoke("How is artificial intelligence being used in business?"))

Artificial intelligence (AI) is being utilized in various ways within businesses, primarily in the following areas:

1. **Human Resources and Hiring**: AI is increasingly being used to manage human employees, particularly in hiring processes. Companies are leveraging AI to sort through resumes, rank candidates, and analyze their responses, as seen with Unilever's use of AI to enhance their hiring efficiency (TechRepublic).

2. **Workforce Management**: AI tools are helping businesses make more objective decisions based on data, which can improve management practices. This includes the potential for AI to assist in identifying employee potential and optimizing workforce configurations (TechRepublic).

3. **Customer Service**: AI is enhancing customer service by providing support to human agents, processing natural language, and improving response times (TechRepublic).

4. **Optimizing Operations**: Businesses are using AI to analyze complex data for various applications, such as farming

## What you can do with this

One AgensgraphVector store gives you, over the same chunks:

- **semantic search** (HNSW) with relevance scores + metadata,
- **metadata-filtered** retrieval (dates, domains, ranges),
- **hybrid** vector+keyword retrieval (RRF, tunable with `fusion_weights`),
- **recall tuning** via `ef_search=N`,
- a drop-in LangChain **retriever** for any LCEL RAG chain or agent.

```python
chain.invoke("your question here")
vector.similarity_search("a topic", k=8, filter={"date": {"$gte": "2018-01-01"}})
```

Close the shared pool when done: `agens.close()`